# AIC-2026 — MetaCLIP 2 keyframe embeddings

Notebook Colab độc lập cho GPU A100 40 GB. Pipeline clone project và source MetaCLIP, đọc trực tiếp hai shared Drive folders, stream từng ZIP để embed ngay, lưu shard có resume, xóa dữ liệu tạm, rồi ghép ma trận cuối.

Checkpoint mặc định: `facebook/metaclip-2-worldwide-huge-quickgelu`. Vector được L2-normalize và lưu `float16`; output nằm trong Google Drive để không mất khi runtime ngắt. Chọn **Runtime → Change runtime type → A100 GPU** trước khi chạy toàn bộ.

In [ ]:
# 1) Clone project và implementation chính thức
import os
from pathlib import Path

PROJECT_URL = "https://github.com/AIVIETNAM-AIO-Dewey/AIC-2026.git"
PROJECT_BRANCH = "feat/keyframe-youtube-mapping"
PROJECT_DIR = Path("/content/AIC-2026-keyframe-embeddings")
METACLIP_REPO = Path("/content/MetaCLIP")

if not (PROJECT_DIR / ".git").exists():
    !git clone --depth 1 --branch {PROJECT_BRANCH} --single-branch {PROJECT_URL} {PROJECT_DIR}
else:
    !git -C {PROJECT_DIR} fetch origin {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} switch {PROJECT_BRANCH}
    !git -C {PROJECT_DIR} pull --ff-only origin {PROJECT_BRANCH}

if not (METACLIP_REPO / ".git").exists():
    !git clone --depth 1 https://github.com/facebookresearch/MetaCLIP.git {METACLIP_REPO}
else:
    !git -C {METACLIP_REPO} pull --ff-only

%cd {PROJECT_DIR}

In [ ]:
# 2) Cài dependency tương thích Colab. Không thay PyTorch/CUDA do runtime cung cấp.
%pip install -q "requests==2.32.4" "huggingface-hub==1.29.0" "transformers==5.16.1" "accelerate==1.14.0" google-api-python-client google-auth-httplib2 tqdm Pillow

In [ ]:
# 3) Mount Drive để giữ output và cấp quyền đọc hai shared folders
from google.colab import auth, drive

drive.mount("/content/drive")
auth.authenticate_user()
print("Google Drive authentication ready")

In [ ]:
# 4) Cấu hình chạy
from pathlib import Path
import shutil
import torch

DATA_ROOT_FOLDER_IDS = [
    "1ZZpoqN-gehvKO9jUG45Gr3wFIcIcbAta",  # 434 ZIP + 434 map CSV
    "1_8Y2PWseN5Max2Lwi43_DW_ulo-T-NzE",  # 439 ZIP + 439 map CSV
]
DATA_DIR = Path("/content/aic_keyframes")          # SSD local: đọc ảnh nhanh
ZIP_CACHE_DIR = Path("/content/aic_zip_cache")     # mỗi ZIP bị xóa sau khi giải nén
OUTPUT_DIR = Path("/content/drive/MyDrive/AIC-2026/embeddings/metaclip2")
STREAM_FROM_SHARED_DRIVE = True                       # tải 1 ZIP → embed ngay → xóa tạm
BATCH_SIZE = 128                                      # giảm còn 64 nếu runtime không phải A100 40 GB
NUM_WORKERS = 4
MAX_ZIPS = None                                       # đặt 2 để smoke test, None để chạy đủ 873 ZIP

assert torch.cuda.is_available(), "Hãy bật GPU runtime trong Colab"
free_gib = shutil.disk_usage("/content").free / 1024**3
print(torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GiB")
print(f"Local disk còn {free_gib:.1f} GiB")
if not STREAM_FROM_SHARED_DRIVE and MAX_ZIPS is None and free_gib < 40:
    raise RuntimeError("Cần tối thiểu khoảng 40 GiB local disk trống cho toàn bộ keyframe")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# 5) Tải dữ liệu + embed. Có thể chạy lại: shard video hợp lệ sẽ được bỏ qua.
import subprocess
import sys
from collections import deque

SCRIPT_PATH = PROJECT_DIR / "scripts" / "colab_keyframe_embeddings.py"
if not SCRIPT_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy {SCRIPT_PATH}. Hãy commit + push file scripts/colab_keyframe_embeddings.py "
        "lên đúng GitHub repo/branch rồi chạy lại cell clone."
    )

command = [
    sys.executable,
    str(SCRIPT_PATH),
    "--model", "metaclip2",
    "--data-dir", str(DATA_DIR),
    "--zip-cache-dir", str(ZIP_CACHE_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
]
for folder_id in DATA_ROOT_FOLDER_IDS:
    command += ["--data-root-folder-id", folder_id]
if STREAM_FROM_SHARED_DRIVE:
    command += ["--stream-archives"]
if MAX_ZIPS is not None:
    command += ["--max-zips", str(MAX_ZIPS), "--allow-count-mismatch"]
print("Running:", " ".join(command), flush=True)
process = subprocess.Popen(
    command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
log_tail = deque(maxlen=120)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
    log_tail.append(line)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(
        f"Embedding process failed with exit code {return_code}. Log tail:\n"
        + "".join(log_tail)
    )

In [ ]:
# 6) Kiểm tra artifact cuối
import csv
import json
import numpy as np

manifest = json.loads((OUTPUT_DIR / "run_manifest.json").read_text(encoding="utf-8"))
vectors = np.load(OUTPUT_DIR / "keyframes_visual_vectors.f16.npy", mmap_mode="r")
metadata_count = sum(1 for _ in (OUTPUT_DIR / "keyframes_metadata.jsonl").open(encoding="utf-8"))
with (OUTPUT_DIR / "keyframe_index.csv").open(encoding="utf-8", newline="") as handle:
    index_reader = csv.DictReader(handle)
    first_mapping = next(index_reader)
    index_count = 1 + sum(1 for _ in index_reader)
sample = vectors[np.linspace(0, len(vectors) - 1, min(1000, len(vectors)), dtype=int)].astype(np.float32)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("matrix:", vectors.shape, vectors.dtype)
print("metadata/index rows:", metadata_count, index_count)
print("first row mapping:", first_mapping)
print("sample norm range:", np.linalg.norm(sample, axis=1).min(), np.linalg.norm(sample, axis=1).max())
assert vectors.shape[0] == metadata_count == index_count == manifest["keyframe_count"]
assert vectors.shape[1] == 1024
assert int(first_mapping["global_vector_row"]) == 0
assert int(first_mapping["point_id"]) == 1
assert first_mapping["frame_idx"] != ""
assert first_mapping["pts_time_s"] != ""
assert first_mapping["fps"] != ""
assert np.isfinite(sample).all()
assert np.allclose(np.linalg.norm(sample, axis=1), 1.0, atol=2e-3)
print("✅ MetaCLIP 2 artifacts verified")

## Output

- `shards/Lxx_Vxxx.f16.npy`: checkpoint/resume theo video.
- `keyframes_visual_vectors.f16.npy`: ma trận toàn bộ keyframe theo thứ tự metadata.
- `keyframes_metadata.jsonl`: metadata máy đọc, cùng thứ tự với ma trận toàn cục.
- `keyframe_index.csv`: mapping chính xác gồm `global_vector_row`, `point_id`, `video_id`, `keyframe_n`, `frame_idx`, `pts_time_s`, `fps`, shard row và filename.
- `keyframe_maps_snapshot.json`: cache CSV map từ hai data root để lần chạy lại không tải lại 873 file map nhỏ.
- `drive_archives_manifest.json`: snapshot 873 ZIP đầu vào.
- `run_manifest.json`: model, dimension, preprocessing, count và GPU.

Ở chế độ streaming, ảnh chỉ tồn tại tạm trong lúc embed một video; shard hoàn tất trên MyDrive giúp lần chạy lại bỏ qua cả bước tải ZIP. Nếu đổi checkpoint/model, hãy dùng một `OUTPUT_DIR` mới để không trộn shard khác không gian vector.